# From Conversation to Tools: Function Calling and Agents

> Distillation can compress abilities a model already has, but it cannot place current external information into the weights automatically. Live weather belongs in an API, exact multiplication is better delegated to a calculator, and repository statistics require reading Git. When an answer lives outside the model, continuing to generate text cannot retrieve it reliably.
>
> This chapter follows one path through three stages. First, learn to **use** tools: `model emits call -> external program executes -> result returns to conversation -> model writes answer`. Second, learn to **teach** this behavior: valid JSON requires purpose-built examples and token-level loss masks. Third, allow the model to **act**: place calling, execution, feedback, and another decision in a loop, and let the model decide when to stop. That loop is an Agent.
>
> Along the way we cover message structures, Chat Templates, token accounting, common failures, and the standardization path from Function Calling to Tool Use and MCP.


The question “what is 357 x 289?” belongs to a calculator; “what is the weather in Beijing now?” requires a live API. A model may understand both requests, but its parameters alone cannot guarantee exact arithmetic or changing external facts.

We proceed in three steps. First, trace Function Calling data flow: the model produces a function name and structured arguments, a program executes it, the result returns to the conversation, and the model writes a natural-language answer. Second, examine training examples and which tokens receive loss. Third, place the same path in a loop so the model chooses the next step and stopping point, producing an Agent.

```text
user request -> function and arguments -> program execution -> tool result -> final answer
```

We begin with two typical failure modes.


## 1. What Function Calling Provides

An LLM's knowledge comes from training data, and training data has a temporal cutoff. Asked "what is the temperature in Beijing today", the model gives a vague answer based on weather statistics seen during training, but it cannot possibly know the real weather today. This is the first limitation — knowledge cutoff.

The second limitation is numerical computation. The fundamental operations of a Transformer are matrix multiplication and softmax, which are not suited to precise multi-digit multiplication. Asking the model to directly compute 357 × 289 often produces a wrong answer. The model's answer looks reasonable (the order of magnitude is right, the last digit may be right), but the middle digits are often incorrect.

The code below simulates these two failures.


In [ ]:
# Simulate two typical LLM failures: knowledge cutoff + numerical computation errors

# Scenario 1: knowledge cutoff
# The model's training data is in the past, so it cannot know "today's" real data
def fake_llm_today_weather():
    # An LLM that does not understand tools is asked about today's weather
    return "I cannot query Beijing's weather in real time, but the temperature this season is usually between 20-30C."

print("=== Failure 1: knowledge cutoff ===")
print("User: What is the temperature in Beijing today?")
print(f"Model: {fake_llm_today_weather()}")
print("Problem: the model does not know today's real weather and can only give a vague guess")
print()

# Scenario 2: numerical computation error
# Simulate a "model" that gets arithmetic wrong — it outputs an answer that looks reasonable but is wrong
def fake_llm_multiply(a, b):
    # Simulate an LLM computing multiplication; it may get it wrong
    correct = a * b
    wrong = correct + 1234  # deliberately give a wrong answer
    return wrong

a, b = 357, 289
correct = a * b
llm_answer = fake_llm_multiply(a, b)
print("=== Failure 2: numerical computation error ===")
print(f"User: what is {a} x {b}?")
print(f"Model: {llm_answer}")
print(f"Correct: {correct}")
print(f"Gap: the model answer exceeds the correct answer by {llm_answer - correct}")
print()
print('Conclusion')


## 2. The Complete Function Call Flow

The core idea of Function Call is to separate "execution" from "decision":

- Decision (whether to call a tool, which one, what arguments to pass) is done by the model
- Execution (actually querying the weather, doing arithmetic, querying a database) is done by an external program

The complete flow has six steps. Let us walk through a concrete example: the user asks "What is the temperature in Beijing today?".

Step 1: define the tools. Each tool is a JSON description telling the model what the tool is called, what it does, and what parameters it needs.
Step 2: inject the tool description into the system prompt. The model sees this description before generating its reply.
Step 3: the model generates a reply. If it decides a tool is needed, instead of natural language it outputs a structured piece of JSON.
Step 4: the external program parses this JSON and calls the corresponding function.
Step 5: the execution result is wrapped as a new conversation message and added to the conversation history.
Step 6: the model generates the final natural-language reply based on the complete conversation history (including the tool result).


In [ ]:
# Step 1: define the available tools
# Each tool is a dict with three parts: name / description / parameters

import json

tools = [
    {
        "name": "get_weather",
        "description": "Query the current weather for a given city",
        "parameters": {
            "city": {"type": "string", "description": "City name"},
            "unit": {"type": "string", "enum": ["celsius", "fahrenheit"], "description": "Temperature unit"}
        }
    },
    {
        "name": "calculate",
        "description": "Evaluate a mathematical expression",
        "parameters": {
            "expression": {"type": "string", "description": "Mathematical expression"}
        }
    }
]

print("=== Defined tools ===")
for t in tools:
    print(f"\nTool name: {t['name']}")
    print(f"  Description: {t['description']}")
    print(f"  Parameters:")
    for pname, pinfo in t['parameters'].items():
        print(f"    {pname} ({pinfo['type']}): {pinfo['description']}")

In [ ]:
# Step 2: assemble the tool descriptions into the system prompt
# This text is visible to the model, which uses it to decide whether and which tool to call

def build_system_prompt(tools):
    # Build a system prompt from the tool list
    lines = ["You are an assistant that can use the following tools. When external information is needed, output a call request in JSON format.", ""]
    lines.append("Available tools:")
    for t in tools:
        lines.append(f"\nTool: {t['name']}")
        lines.append(f"  Description: {t['description']}")
        lines.append(f"  Parameters:")
        for pname, pinfo in t['parameters'].items():
            lines.append(f"    - {pname}: {pinfo['description']}")
    return "\n".join(lines)

system_prompt = build_system_prompt(tools)
print("=== System Prompt (the model sees this) ===")
print(system_prompt)

In [ ]:
# Step 3: simulate the model generating a JSON call request
# In practice this is the token sequence produced by the model's forward pass; here we hand-simulate a reasonable output

user_query = "What is the temperature in Beijing today?"

# Suppose that, after training, the model faced with this user query and the tool list above outputs this JSON
model_output = json.dumps({
    "tool_call": {
        "name": "get_weather",
        "arguments": {"city": "Beijing", "unit": "celsius"}
    }
}, ensure_ascii=False, indent=2)

print('"=== ModelOutput ==="')
print(f"User: {user_query}")
print("Instead of natural language, the model outputs JSON:")
print(model_output)
print()
print("Key observation: the model needs external information, selects get_weather, and fills city=Beijing.")


In [ ]:
# Step 4 + Step 5: an external program executes the tool, and the result is appended to the conversation history

# Simulate the real implementation of the tools
def execute_tool(name, arguments):
    # Execute the corresponding function based on the tool name
    if name == "get_weather":
        # In practice this would call a real weather API
        return {"temperature": 22, "condition": "sunny", "humidity": 45}
    elif name == "calculate":
        # In practice this would call a real calculator
        return {"result": eval(arguments["expression"])}
    return {"error": f"Unknown tool: {name}"}

# Parse the model's JSON output and execute the corresponding tool
parsed = json.loads(model_output)
call = parsed["tool_call"]
tool_result = execute_tool(call["name"], call["arguments"])

print("=== Tool execution ===")
print(f"Call: {call['name']}({call['arguments']})")
print(f"Return: {tool_result}")
print()

# Build the full conversation history, adding the tool result as a new message
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_query},
    {"role": "assistant", "content": model_output},
    {"role": "tool", "name": call["name"], "content": json.dumps(tool_result, ensure_ascii=False)},
]

print("=== Complete dialogue history, now including the tool result ===")
for m in messages:
    role = m["role"]
    content = m["content"]
    if len(content) > 80:
        content = content[:80] + "..."
    print(f"[{role}] {content}")


In [ ]:
# Step 6: the model generates the final natural-language reply based on the full conversation history

# Simulate the model's final reply
final_reply = "Beijing is 22C and sunny today."

print("=== Final reply ===")
print(f"Model: {final_reply}")
print()
print("Key observation: after seeing temperature 22 and sunny weather, the model combines both into natural language.")
print("What the user perceives is an ordinary answer; the details of the tool call are hidden.")


## 3. Multi-Tool Scenarios

In real scenarios, a single question often needs multiple tools. For example, the user asks "What is the temperature in Beijing today? Also, what is 357 x 289?", and the model needs to call both get_weather and calculate.

There are two cases here, handled differently.

Parallel calls: the two tools have no dependency between them, so order does not affect the result. The model can generate multiple call requests in a single output, the external program executes them concurrently, and all results are fed back at once. This has lower latency.

Sequential calls: the arguments of the second tool depend on the result of the first. The model must call the first tool, see the result, and only then decide whether to call the second and what arguments to pass. For example, "check the temperature in Beijing today, then tell me what to wear" — the second call (clothing recommendation) depends on the result of the first call (weather). This case must proceed over multiple rounds.


In [ ]:
# Parallel calls: generate multiple tool call requests at once

user_query = "What is the temperature in Beijing today? Also, what is 357 x 289?"

# The model outputs multiple calls in one go
model_output = json.dumps({
    "tool_calls": [
        {"name": "get_weather", "arguments": {"city": "Beijing", "unit": "celsius"}},
        {"name": "calculate", "arguments": {"expression": "357*289"}}
    ]
}, ensure_ascii=False, indent=2)

print("=== Parallel call ===")
print(f"User: {user_query}")
print('"ModelOutput："')
print(model_output)
print()

# The external program parses and executes them one by one (in practice they can run concurrently)
parsed = json.loads(model_output)
for call in parsed["tool_calls"]:
    result = execute_tool(call["name"], call["arguments"])
    print(f"Execute {call['name']} -> {result}")

print()
print("Key observation: the model emits two independent calls in one response.")
print("The external program can execute both calls in parallel and return both results to the model together.")


In [ ]:
# Sequential calls: the second call depends on the result of the first

user_query = "What is the temperature in Beijing today? What should I wear?"

# Round 1: the model first calls the weather tool
first_call = json.dumps({
    "tool_call": {"name": "get_weather", "arguments": {"city": "Beijing", "unit": "celsius"}}
}, ensure_ascii=False)

print("=== Sequential call ===")
print(f"User: {user_query}")
print("\nRound 1 model output:")
print(first_call)

weather_result = execute_tool("get_weather", {"city": "Beijing", "unit": "celsius"})
print(f"\nTool result: {weather_result}")
print()

# Round 2: after seeing the weather result, the model decides to call a clothing-suggestion tool
wardrobe_tool_result = {"suggestion": "Long-sleeve shirt with a light jacket"}
print("Round 2 model output: based on weather 22C sunny, call the clothing-suggestion tool")
print(f"Tool result: {wardrobe_tool_result}")
print()
print("Key observation: the second call depends on the first result; the model needs the temperature before suggesting clothing.")
print("This dependency means a sequential call must span multiple rounds, with higher latency than parallel calls.")
print("Between rounds, external code only executes and returns results; the model decides the next call after reading them.")
print("Put this process in a loop and let the model decide when to stop, and you have the Agent at the end of this chapter.")


## 4. Message Structure and Chat Templates

The preceding workflow passes a `messages` list between components. This is the application-layer representation of a conversation. Internally, a language model only predicts continuation and does not inherently know what a “user” or “tool” is. The application must distinguish speakers, global instructions, and tool results. Each message dictionary therefore records a role and content. We will inspect those fields, render them for the model, and count the resulting tokens.


In [ ]:
# Example dialogue for this chapter; later Token accounting and training samples reuse it
sample_messages = [
    {"role": "system", "content": "You are an assistant with tools: get_weather(city) for weather and calculate(expression) for arithmetic."},
    {"role": "user", "content": "What is the temperature in Beijing today?"},
    {"role": "assistant", "content": "",
     "tool_call": {"name": "get_weather", "arguments": {"city": "Beijing", "unit": "celsius"}}},
    {"role": "tool", "name": "get_weather", "content": '{"temperature": 22, "condition": "sunny"}'},
    {"role": "assistant", "content": "It is 22°C and sunny in Beijing today."},
]

print("=== Defined tools ===")
for m in sample_messages:
    body = json.dumps(m["tool_call"], ensure_ascii=False) if m.get("tool_call") else m["content"]
    print(f"[{m['role']:9s}] {body[:56]}")
print()
print("Five turns: system -> user -> assistant tool call -> tool result -> assistant final response.")


### 4.1 Message Structure

A message has two main fields. `role` identifies the speaker as `system`, `user`, `assistant`, or `tool`; `content` contains the message text. The `sample_messages` example below makes their meanings concrete.


### 4.2 Chat Template

The model does not read the messages JSON directly. At inference time, the framework uses a chat template to render the message list into a single continuous piece of text, with special tokens marking the boundary and role of each message. This step is usually done by `tokenizer.apply_chat_template()` and was covered in the Part 3 decoding-strategy section.

Using a Qwen-style template as an example, two messages (system and user) render roughly like this:

```text
<|im_start|>system
You are an assistant that can use the following tools: get_weather(city)...<|im_end|>
<|im_start|>user
What is the temperature in Beijing today?<|im_end|>
<|im_start|>assistant
```

`<|im_start|>role` marks the start of a given role's turn, and `<|im_end|>` marks the end of the message — these are the special tokens covered in Part 1, each with a fixed ID in the vocabulary. The trailing unclosed `<|im_start|>assistant` hands the turn to the model, which starts generating from there. Different models use different templates: LLaMA 3 uses `<|start_header_id|>`, and the GPT family uses something like `<|im_sep|>`. Using the wrong template means the model reads a format inconsistent with what it saw in training, and reply quality drops noticeably — which is exactly why `apply_chat_template()` exists: it remembers each model's own format.

### 4.3 Token Count and Context Length

The boundary matches the loss mask in Section 5.1. During training, system/user/tool tokens are inputs and usually excluded from loss, while assistant tokens are outputs and receive loss. During inference accounting, the question changes from “does this token contribute to loss?” to “is this an input or output token?”, but the boundary remains the same.

The next cell uses `tiktoken` to count input and output tokens in `sample_messages`.


In [ ]:
# Use tiktoken to actually count the tokens of the Section 4 conversation
# cl100k_base is the encoding used by the GPT-3.5/4 family; with a different tokenizer, the numbers change
try:
    import tiktoken
    enc = tiktoken.get_encoding("cl100k_base")
    def n_tokens(text):
        return len(enc.encode(text))
except Exception:
    # Fall back to a rough character estimate (Chinese ~1.5 chars/token) when tiktoken is not installed
    print('Number of tokens: ')
    def n_tokens(text):
        return max(1, round(len(text) / 1.5))

# Same boundary as the loss mask in 4.1: assistant counts as output, the rest as input
input_tokens = 0
output_tokens = 0
print("Counting tokens per message:")
for msg in sample_messages:
    role = msg["role"]
    # The assistant's call intent is in tool_call and must be counted as output
    if role == "assistant" and msg.get("tool_call"):
        text = json.dumps(msg["tool_call"], ensure_ascii=False)
    else:
        text = msg["content"]
    cnt = n_tokens(text)
    if role == "assistant":
        output_tokens += cnt
        tag = '"Output"'
    else:
        input_tokens += cnt
        tag = '"Input"'
    print(f"  [{role:9s}] {tag} | {cnt:3d} token | {text[:24]}")

print()
print(f"input tokens (system + user + tool) = {input_tokens}")
print(f"output tokens (assistant)           = {output_tokens}")
print(f"context used = input + output       = {input_tokens + output_tokens}")

In [ ]:
# After reordering, this cell runs before the training section, so import plt locally
import matplotlib.pyplot as plt
# === Visualize the composition of input and output Tokens ===
# Count each message again, separating input and output by role
counts, labels, colors = [], [], []
for msg in sample_messages:
    role = msg["role"]
    if role == "assistant" and msg.get("tool_call"):
        text = json.dumps(msg["tool_call"], ensure_ascii=False)
    else:
        text = msg["content"]
    counts.append(n_tokens(text))
    labels.append(role)
    colors.append("#f97316" if role == "assistant" else "#94a3b8")

plt.figure(figsize=(8, 3.5))
plt.barh(range(len(counts)), counts, color=colors)
plt.yticks(range(len(counts)), labels)
plt.gca().invert_yaxis()
plt.xlabel("Tokens")
plt.title("Token count per message (orange = supervised output)")
plt.tight_layout()
plt.show()


**Three reasons input and output are listed separately**

First, different compute patterns lead to different pricing. Input goes through prefill, computed in parallel — fast and cheap; output goes through decode, generated one token at a time and bandwidth-heavy — slow. So across API providers the unit price of output tokens is higher, typically 3 to 4 times that of input.

Second, different limits lead to different constraints. Many models cap output well below the context length — for example, a 128K context but only 4K or 8K of maximum output. Decode is inherently slow, and the KV cache grows linearly with length in memory (concrete numbers were computed in the Part 4 long-context section), both of which limit how much can be generated in one request.

Third, different optimizability. When multiple requests share the same system message (in this section, every request carries the same tool description), prefix caching can reuse the KV for this input stretch, cutting its cost by about 90% on a hit (covered in the Part 3 inference-systems section); output cannot be reused — every step is recomputed.

The usage field returned by the API is the billing basis: `prompt_tokens` is the input tokens and `completion_tokens` is the output tokens. The `max_model_len` in vLLM deployment limits the sum of input plus output (Part 5 deployment section).

## 5. Training a Model for Function Calling

The inference mechanism is complete: messages describe the conversation, a Chat Template renders them, the model emits tool-call JSON, and an external program executes it and returns the result. A pretrained model does not reliably produce this format by itself; it may generate almost-JSON with invalid syntax or invented arguments.

Reliable behavior requires examples that teach the full path: understand a tool description, emit a valid call, and finish the answer from the result. The format resembles instruction tuning, except the assistant appears twice—once for the tool call and once for the final response. We follow one sample down to token-level loss.


In [ ]:
# One complete Function Call training sample: the dialogue itself is sample_messages from Section 4
# Only the viewpoint changes: treat it as training data and identify which parts the model must learn to generate
training_sample = {"messages": sample_messages}

print("=== A Function Call training sample ===")
print(json.dumps(training_sample, ensure_ascii=False, indent=2))
print()
print("This sample contains the full flow: system -> user -> assistant tool call -> tool result -> assistant final response.")
print("Many such samples teach the model when to emit tool-call JSON and how to answer after receiving a result.")


In [ ]:
# A complete Function Call training sample

training_sample = {
    "messages": [
        {
            "role": "system",
            "content": "You are an assistant that can use the following tools: get_weather(city) to query the weather; calculate(expression) to compute."
        },
        {
            "role": "user",
            "content": "What is the temperature in Beijing today?"
        },
        {
            "role": "assistant",
            "content": "",
            "tool_call": {"name": "get_weather", "arguments": {"city": "Beijing", "unit": "celsius"}}
        },
        {
            "role": "tool",
            "name": "get_weather",
            "content": '{"temperature": 22, "condition": "sunny"}'
        },
        {
            "role": "assistant",
            "content": "Beijing is 22C and sunny today."
        }
    ]
}

print("=== A Function Call training sample ===")
print(json.dumps(training_sample, ensure_ascii=False, indent=2))
print()
print("This sample contains the full flow: system -> user -> assistant tool call -> tool result -> assistant final response.")
print("With many samples like these, the model learns when to emit tool-call JSON.")


### 5.1 Templates and Token-Level Loss

The training-and-loss notebook walked through the skeleton — render, tokenize, mask — using a single-turn Q&A. A Function Call training sample adds a detail that ordinary conversations do not have: the assistant appears twice, with a tool-result turn in between. So the supervised span is not one contiguous block but two segments — the tool-call JSON and the final reply.

This subsection follows the training sample from the top of this section down to the token level, in four steps, and every step's output can be verified by hand:

```text
Step 1   chat template: messages -> one continuous text with special tokens
Step 2   tokenizer: text -> a token sequence where every token is countable
Step 3   labels: supervised positions keep the token id; input positions become -100
Step 4   cross entropy: each supervised position contributes -log p, averaged over supervised positions only
```

Why is the ignore marker -100 of all things? It is the default `ignore_index` of PyTorch `cross_entropy`: positions marked -100 enter neither the numerator nor the denominator of the loss. Hand-computing cross entropy itself was done in the training-and-loss notebook; here we walk the masked pipeline in pure Python. Real training libraries do exactly the same thing, only with a real tokenizer and template.

In [ ]:
# Step 1: render messages into one continuous string with a simplified chat template
# Qwen-style format: <|im_start|>role\n content <|im_end|>\n

# Real Qwen templates also wrap tool-call JSON in <tool_call> tags; this example omits them
def apply_simple_template(sample):
    """Render messages into text segments paired with a flag indicating whether each segment contributes loss."""
    segments = []
    for msg in sample["messages"]:
        role = msg["role"]
        # The role marker at each turn tells the model whose turn it is; it is input and receives no loss
        segments.append((f"<|im_start|>{role}\n", False))
        if role == "assistant" and msg.get("tool_call"):
            # Tool-call JSON is output the model must learn to generate, so it receives loss
            body = json.dumps(msg["tool_call"], ensure_ascii=False)
            segments.append((body, True))
        elif role == "assistant":
            # The final response is also model output and receives loss
            segments.append((msg["content"], True))
        else:
            # System instructions, user questions, and tool results are inputs
            segments.append((msg["content"], False))
        # The assistant must learn to stop, so its <|im_end|> receives loss; endings for other roles are inputs
        segments.append(("<|im_end|>", role == "assistant"))
        # The newline after an ending is only formatting and remains input
        segments.append(("\n", False))
    return segments

segments = apply_simple_template(training_sample)
full_text = "".join(text for text, _ in segments)

print("=== Step 1: complete training text rendered by the chat template ===")
print(full_text)
print("--- Key observation ---")
print("Loss applies only to assistant tool-call JSON and the assistant final response.")
print("The <|im_start|>assistant at each turn is input: the model learns what to output next,")
print("not that it is now the assistant's turn.")
print("Both assistant <|im_end|> markers also contribute loss so the model learns when to stop.")


In [ ]:
# Step 2: tokenize the text with a transparent simplified tokenizer
# It follows only three rules, making every Token countable by hand:
#   1. Treat <|im_start|> and <|im_end|> as one Token each
#   2. Join consecutive letters, digits, and underscores into one Token, such as get_weather, 22, celsius
#   3. Treat every other character, including punctuation, spaces, and newlines, as one Token
SPECIALS = ["<|im_start|>", "<|im_end|>"]

def simple_tokenize(text):
    """Split a piece of text into a character-level token list"""
    tokens = []
    i = 0
    while i < len(text):
        # Rule 1: one whole special Token
        hit = next((s for s in SPECIALS if text.startswith(s, i)), None)
        if hit is not None:
            tokens.append(hit)
            i += len(hit)
        # Rule 2: join English words and numbers
        elif text[i].isascii() and (text[i].isalnum() or text[i] == "_"):
            j = i
            while j < len(text) and text[j].isascii() and (text[j].isalnum() or text[j] == "_"):
                j += 1
            tokens.append(text[i:j])
            i = j
        # Rule 3: split remaining characters one by one
        else:
            tokens.append(text[i])
            i += 1
    return tokens

# Carry the loss mask through tokenization: every Token from a supervised span receives loss
token_list, loss_mask = [], []
for seg_text, seg_loss in segments:
    seg_tokens = simple_tokenize(seg_text)
    token_list.extend(seg_tokens)
    loss_mask.extend([seg_loss] * len(seg_tokens))

# Build a small vocabulary: special Tokens first, then others in first-appearance order
vocab = {s: k for k, s in enumerate(SPECIALS)}
for t in token_list:
    if t not in vocab:
        vocab[t] = len(vocab)
input_ids = [vocab[t] for t in token_list]

print("=== FFN does not mix tokens ===")
print(f"The training text contains {len(token_list)} Tokens; vocabulary size is {len(vocab)}")
print('Vocabulary')
print()
print("Interpretation:")
print("  role header:", simple_tokenize("<|im_start|>assistant\n"))
print("  user message:", simple_tokenize("What is the temperature in Beijing today?"))
print("  final reply:", simple_tokenize("It is 22°C and sunny in Beijing today."))
print()
print("Note: spaces and newlines are Tokens; JSON quotes and braces are Tokens too.")
print("This is why tool-call output uses more tokens than it appears to; Section 5.3 counts them directly.")


In [ ]:
# Step 3: build labels—keep Token IDs at supervised positions and set input positions to -100
labels = [tid if m else -100 for tid, m in zip(input_ids, loss_mask)]

# Find contiguous supervised runs first, then mark their boundaries in the per-Token table
runs = []
start = None
for i, m in enumerate(loss_mask + [False]):
    if m and start is None:
        start = i
    elif not m and start is not None:
        runs.append((start, i - 1))
        start = None
run_names = ["tool-call JSON", "final response"]

print("=== Step 3: per-Token labels; label = -100 means ignored ===")
run_no = -1
for i, (tok, lab) in enumerate(zip(token_list, labels)):
    # Insert a marker when entering a supervised run
    if run_no + 1 < len(runs) and i == runs[run_no + 1][0]:
        run_no += 1
        print(f"------ supervised run {run_no + 1} begins ({run_names[run_no]}) ------")
    print(f"{i:3d} | {repr(tok):>14} | {lab:4d}")
    # Print a summary when leaving a supervised run
    if run_no >= 0 and i == runs[run_no][1]:
        print(f"------ supervised run {run_no + 1} ends; {i - runs[run_no][0] + 1} Tokens ------")

n_valid = sum(1 for l in labels if l != -100)
print()
print(f"There are {len(labels)} positions: {n_valid} receive loss and {len(labels) - n_valid} are -100.")
print("The two supervised spans are separated by the entire tool-result turn; this is the defining mask pattern")
print("This is the main difference from an ordinary single-turn SFT sample.")


In [ ]:
# Visualize the entire Token sequence as a timeline so the two supervised regions are immediately visible
# Figure text is English under the project rules
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

run_colors = ["#2a9d8f", "#e9c46a"]  # one color per supervised region
fig, ax = plt.subplots(figsize=(12, 2.8))
for i in range(len(token_list)):
    color = "#d9d9d9"  # gray by default means label is -100
    for r, (a, b) in enumerate(runs):
        if a <= i <= b:
            color = run_colors[r]
    ax.barh(0, 1, left=i, height=1, color=color, edgecolor="white", linewidth=0.4)

# Mark each message span above; every message begins at an <|im_start|>
msg_starts = [i for i, t in enumerate(token_list) if t == "<|im_start|>"] + [len(token_list)]
for k in range(len(msg_starts) - 1):
    a, b = msg_starts[k], msg_starts[k + 1] - 1
    role = token_list[a + 1]
    ax.plot([a, b + 1], [1.45, 1.45], color="#555555", linewidth=1.2)
    ax.plot([a, a], [1.3, 1.6], color="#555555", linewidth=1.2)
    ax.plot([b + 1, b + 1], [1.3, 1.6], color="#555555", linewidth=1.2)
    ax.text((a + b + 1) / 2, 1.75, role, ha="center", va="bottom", fontsize=9)

# Annotate Token counts beneath supervised regions
for r, (a, b) in enumerate(runs):
    ax.text((a + b) / 2, -1.15, f"{b - a + 1} tokens", ha="center", va="top",
            fontsize=9, color=run_colors[r])

ax.set_xlim(-1, len(token_list) + 1)
ax.set_ylim(-1.8, 2.4)
ax.set_yticks([])
ax.set_xlabel("Token position")
ax.set_title("Token-level loss mask of one function-calling training sample")
legend_items = [Patch(facecolor="#d9d9d9", label="label = -100 (ignored)"),
                Patch(facecolor=run_colors[0], label="supervised: tool-call JSON"),
                Patch(facecolor=run_colors[1], label="supervised: final reply")]
ax.legend(handles=legend_items, loc="upper center", ncol=3,
          bbox_to_anchor=(0.5, -0.35), frameon=False)
plt.tight_layout()
plt.show()


Before step 4, two mechanisms need to be clear.

**shift: the prediction target of position i is the token at position i+1.** labels is as long as input_ids and aligned with it; the one-position shift is applied inside the model (the training-and-loss notebook compared the two equivalent ways: shifting in the data or inside the model). The mask therefore lands on labels — what is supervised is the *target token*. A concrete example: the first token of supervised segment 1 is `{` (position 71), and the context predicting it ends exactly at `<|im_start|>assistant` plus the newline — precisely where the model starts generating at inference time.

**-100: skipped, and kept out of the denominator.** Each supervised position contributes $-\log p(\text{target token} \mid \text{prefix})$, and the total loss is the sum over supervised positions divided by the *number of supervised positions*. The denominator is not the sequence length: this sample has 150 positions but only 51 supervised ones. Averaging over 150 would dilute the loss with input positions and weaken the training signal of "what to output where".

In PyTorch the two mechanisms collapse into one line: `F.cross_entropy(shift_logits, shift_labels, ignore_index=-100)`.

In [ ]:
# Step 4: compute Cross-Entropy at every position, then average only supervised positions
# Real training obtains logits from model.forward; here seeded random numbers simulate one row of logits
# Their values do not matter; the point is to see how positions with -100 are skipped
import math
import random

random.seed(42)

def softmax(xs):
    """Turn a row of logits into probabilities, subtracting the maximum to avoid exp overflow."""
    m = max(xs)
    exps = [math.exp(x - m) for x in xs]
    return [e / sum(exps) for e in exps]

# Create one logit row per prediction position: logits at i predict labels[i + 1]
n_predict = len(labels) - 1
all_logits = [[random.gauss(0, 1) for _ in range(len(vocab))] for _ in range(n_predict)]

per_token_loss = []
for i in range(n_predict):
    target = labels[i + 1]
    if target == -100:
        per_token_loss.append(None)  # None marks a position skipped by the mask
        continue
    p_correct = softmax(all_logits[i])[target]
    per_token_loss.append(-math.log(p_correct))

# Hand-calculate the first supervised position to see where one position loss comes from
first_pos = next(i for i, l in enumerate(per_token_loss) if l is not None)
target = labels[first_pos + 1]
probs = softmax(all_logits[first_pos])
print("=== Per-position computation ===")
print(f"Logits at position {first_pos} predict position {first_pos + 1}, whose Token is {repr(token_list[first_pos + 1])}")
print(f"First five logits out of {len(vocab)}: {[round(x, 2) for x in all_logits[first_pos][:5]]}")
print(f"Softmax probability of the correct Token p = {probs[target]:.4f}")
print(f"Position loss = -log(p) = {-math.log(probs[target]):.4f}")
print(f"Matches per_token_loss[{first_pos}]: {per_token_loss[first_pos]:.4f}")
print()

valid_losses = [l for l in per_token_loss if l is not None]
print(f"There are {n_predict} prediction positions; {len(valid_losses)} are supervised and the rest are -100")
mean_loss = sum(valid_losses) / len(valid_losses)
print(f"Correct mean, denominator = {len(valid_losses)} supervised positions: {mean_loss:.4f}")
wrong_mean = sum(valid_losses) / n_predict
print(f"Wrong mean, denominator = all {n_predict} prediction positions: {wrong_mean:.4f}")
print()
print(f"Key observation 1: random logits give mean loss around {mean_loss:.1f}, near log(vocabulary size) = "
      f"{math.log(len(vocab)):.2f}; an untrained model guesses every Token blindly.")
print("Key observation 2: the wrong denominator makes loss artificially small; training libraries average over supervised positions.")
print()

# Visualize per-position loss: gray is mask-skipped, green is supervised
plt.figure(figsize=(12, 2.6))
heights = [l if l is not None else 0 for l in per_token_loss]
bar_colors = ["#2a9d8f" if l is not None else "#e0e0e0" for l in per_token_loss]
plt.bar(range(n_predict), heights, color=bar_colors, width=1.0)
plt.axhline(mean_loss, color="#d62828", linestyle="--", linewidth=1.2,
            label=f"mean over supervised = {mean_loss:.2f}")
plt.axhline(wrong_mean, color="#999999", linestyle=":", linewidth=1.2,
            label=f"mean over all positions = {wrong_mean:.2f}")
plt.xlabel("Position i (logits at i predict token i+1)")
plt.ylabel("Loss")
plt.title("Per-position cross-entropy: only supervised positions contribute")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()


### 5.2 Function-Calling Training Data

Constructing training samples is the most time-consuming part of Function Call training. There are three common data sources:

| Source | Approach | Pros and cons |
|:---|:---|:---|
| Human annotation | Annotators see the tool list and write "user question + expected tool call" pairs | High quality, but expensive and hard to scale |
| Strong-model distillation | A model such as GPT-4 that already supports tool calls generates call samples, which are then used as training data | Easy to scale, but capped by the strong model's ability |
| Real logs | Filter correct samples from existing tool-call logs of a product | Closest to the real distribution, but no logs exist during cold start |

In practice, projects usually mix all three: first use human annotation for a few thousand high-quality seed samples, then scale up to tens of thousands via strong-model distillation, and finally incorporate real logs for continuous optimization.


## 6. Error Handling and Failure Modes

A tool call does not always succeed on the first try. Real deployments encounter a variety of failures, and both the model and the external program must be able to handle them.

Below are the four most common failure modes, each paired with a concrete example.

In [ ]:
# Failure mode 1: tool execution failure
# For example, the weather API times out, or the user passes a city that does not exist

def execute_tool_with_failure(name, arguments):
    # A tool executor that may fail
    if name == "get_weather":
        city = arguments.get("city", "")
        # Simulate "city does not exist"
        if city not in ["Beijing", "Shanghai", "Guangzhou"]:
            return {"error": f"City not found: {city}"}
        return {"temperature": 22, "condition": "sunny"}
    return {"error": "Unknown tool"}

# Case: the model called a city that does not exist
result = execute_tool_with_failure("get_weather", {"city": "Atlantis"})
print("=== Failure mode 1: tool execution failure ===")
print("Model call: get_weather(city='Atlantis')")
print(f"Tool return: {result}")
print()
print("Engineering response: return the raw error to the model so it can apologize or ask a clarifying question.")
print("Example: 'Sorry, weather information for Atlantis was not found. Which city did you mean?'")


In [ ]:
# Failure mode 2: the model picks the wrong tool
# The user asked for a computation, but the model called the weather tool

wrong_call = {"name": "get_weather", "arguments": {"city": "357*289"}}
print("=== Failure mode 2: the model picks the wrong tool ===")
print("User: what is 357 x 289?")
print(f"Model call: {wrong_call}")
print("Problem: the model treated the arithmetic expression as a city name")
print()
print("Engineering countermeasure: add more 'user-question type -> correct tool' samples to the training data.")
print("You can also emphasize each tool's applicable scenario in the system prompt.")


In [ ]:
# Failure mode 3: invalid arguments
# For example, the tool requires unit to be only celsius or fahrenheit, but the model passed kelvin

bad_call = {"name": "get_weather", "arguments": {"city": "Beijing", "unit": "kelvin"}}
print("=== Failure mode 3: invalid arguments ===")
print(f"Model call: {bad_call}")
print("Problem: unit='kelvin' is not in enum ['celsius', 'fahrenheit']")
print()

# Engineering countermeasure: strict validation with a JSON Schema
valid_units = ["celsius", "fahrenheit"]
actual_unit = bad_call["arguments"]["unit"]
if actual_unit not in valid_units:
    print(f"Validation failed: unit must be one of {valid_units}, but received '{actual_unit}'")
    print("Engineering countermeasure: add schema validation in the external program; reject invalid arguments outright,")
    print("and feed the validation error back to the model so it can retry.")


In [ ]:
# Failure mode 4: tool-call infinite loop
# The model repeatedly calls the same tool and never outputs a final reply

print("=== Failure mode 4: tool-call infinite loop ===")
print("Scenario: the model calls get_weather several times in a row with identical arguments, never emitting a final reply")
print()

# Engineering countermeasure: set a maximum call count
MAX_TOOL_CALLS = 3
call_count = 0
for i in range(MAX_TOOL_CALLS + 2):  # simulate the model calling repeatedly
    call_count += 1
    if call_count > MAX_TOOL_CALLS:
        print(f"Call {call_count}: exceeded maximum {MAX_TOOL_CALLS}; stopping forcibly")
        break
    print(f"Call {call_count}: get_weather(city='Beijing')")

print()
print("Engineering countermeasure: wrap the call loop with a counter; when the threshold is exceeded, terminate forcibly and return a fallback reply.")
print("Additionally, the training data can include positive samples that 'reply after one call' to reduce the model's tendency to call repeatedly.")


## 7. Standardizing Tool Calls

OpenAI introduced the name Function Calling in June 2023 with a `functions` API parameter. It later generalized the interface to Tools, renamed the parameter to `tools`, and replaced `function_call` with `tool_calls`. The broader design recognizes that a tool may be a retriever, code interpreter, file system, or any callable capability rather than only a function. Tool Use and Tool Calling are therefore now more general terms.

Providers still expose different tool interfaces. Anthropic introduced MCP (Model Context Protocol) in late 2024 to standardize tool descriptions and invocation, allowing compatible tools and model clients to interoperate without a separate adapter for every model.

Standardization answers where tools come from and how they are described. When a task needs many tools, another question remains: who determines call order and when to stop? The Agent loop delegates those choices to the model.


## 8. Toward Agents: Put Tool Calling in a Loop

An **Agent** is a loop in which a model autonomously carries out a multi-step task: it chooses a tool, an external program executes it, the result returns to the model, and the model decides the next step until it concludes the task and produces a final response.

All required pieces have already appeared:

- structured tool calls and external execution;
- serial calls whose later steps depend on earlier results;
- training examples that place tool messages in conversation history;
- error feedback and an outer maximum-turn limit.

```text
while below the maximum number of turns:
    response = model(full conversation history)
    if response contains a tool_call:
        execute it and append the result as a tool message
    else:
        return the final response and stop
```

In ordinary orchestration, external code fixes the number of calls. In an Agent, the stopping signal comes from the model: natural-language output instead of another `tool_call` ends the loop. Training must therefore supervise both calling a tool and finishing after receiving its result.


In [ ]:
# Minimal Agent loop: model, tools, loop, and stop condition
# Rules simulate the model; in a real setting, replace fake_agent_model with one LLM API call

def fake_agent_model(messages):
    """Simulate a tool-capable model: call before seeing a result, then finish after receiving one."""
    tool_msgs = [m for m in messages if m["role"] == "tool"]
    if not tool_msgs:
        # With no tool result, issue the first call
        return {"role": "assistant", "content": "",
                "tool_call": {"name": "get_weather",
                              "arguments": {"city": "Beijing", "unit": "celsius"}}}
    # With a tool result, output the final response without tool_call, stopping the loop
    weather = json.loads(tool_msgs[-1]["content"])
    return {"role": "assistant",
            "content": f"It is {weather['temperature']}°C and {weather['condition']} in Beijing; wear a long-sleeve shirt and light jacket."}

agent_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "What is the temperature in Beijing today, and what should I wear?"},
]

MAX_TURNS = 5  # safeguard: force termination if the model never stops, matching the infinite-loop issue in Section 6
for turn in range(1, MAX_TURNS + 1):
    reply = fake_agent_model(agent_messages)  # one model inference in a real setting
    agent_messages.append(reply)
    if reply.get("tool_call"):
        call = reply["tool_call"]
        print(f"Turn {turn}: model calls {call['name']}({call['arguments']})")
        result = execute_tool(call["name"], call["arguments"])
        agent_messages.append({"role": "tool", "name": call["name"],
                               "content": json.dumps(result, ensure_ascii=False)})
        print(f"        tool returns {result}; append to history and continue")
    else:
        print(f"Turn {turn}: model outputs natural language without tool_call; stop")
        print(f"\nFinal response: {reply['content']}")
        break

print()
print("Key observation: external code only loops and handles fallback; the model output decides what to call and when to stop.")
print("The serial call in Section 3 is simply this loop running for two turns.")


Real-world Agents add three concerns to this skeleton. **Planning** decomposes complex tasks into steps; ReAct alternates a reasoning trace with actions. **Memory** uses conversation history in the short term and external storage plus retrieval across sessions or beyond the context window. **Permissions** require confirmation for side effects such as sending mail or editing files, because one wrong tool choice can be amplified by the loop.

This is why tool reliability is the foundation of an Agent. A 95% per-step success rate falls to about 60% over ten independent steps. Training-data quality and failure handling must ultimately be evaluated inside the complete loop.


## Summary

What this section covered:

- LLMs have two fundamental limitations: knowledge cutoff (cannot know events after training) and numerical computation errors (not good at precise arithmetic)
- Function Call separates "decision" from "execution": the model generates a structured JSON call request, the external program executes it
- The complete six-step flow: define tools -> inject into system prompt -> model outputs JSON -> external execution -> result feedback -> model generates final reply
- Multi-tool scenarios split into two kinds: parallel calls (no dependency, multiple can be generated at once) and sequential calls (with dependency, must span multiple rounds)
- Training a Function Call model needs dedicated samples: a full conversation flow (system + user + assistant tool call + tool result + assistant final reply); the token-level loss mask marks every system/user/tool token as -100 and supervises only the assistant's two output segments (tool-call JSON and final reply), including the closing <|im_end|>
- Three data sources: human annotation, strong-model distillation, real logs
- A message has two core fields, role and content, plus two conditional fields tool_call and name; at inference it is rendered by a chat template into continuous text delimited by special tokens
- Input tokens (system/user/tool) plus output tokens (assistant) must not exceed the context length; input goes through prefill and output through decode, which is why APIs bill them separately
- Four common failure modes: tool execution failure, model picking the wrong tool, invalid arguments, infinite call loop
- Naming evolution: Function Call -> Tool Use -> MCP; behind it is the expansion from "calling functions" to "calling arbitrary tools" to "a standardized tool interface"

## Exercises

Three exercises cover three categories of content:

1. **Core mechanism**: hand-write a tool call JSON to understand the structure of the model's output
2. **Training data**: annotate the loss mask of a training sample to understand which tokens count toward loss
3. **Multi-tool calling**: judge whether a scenario needs parallel or sequential calls to understand dependencies

> You can ask an AI to help explain the approach, but it is not recommended to have the AI complete the exercise directly for you.


### Exercise 1: Hand-write a tool call JSON

Given a user question and a tool list, write the call JSON that the model should output.

Tool list:

- `calculate(expression)`: perform a mathematical computation
- `search(query)`: web search

User question: `What is 123 + 456 x 7?`

**Hint**: the model has to decide which tool to call and what arguments to pass. expression should be a string containing a mathematical expression.


In [ ]:
# Exercise 1: hand-write a tool call JSON
import json

# TODO: replace the placeholder string below with the JSON string you write
# Format: {"tool_call": {"name": "...", "arguments": {...}}}
your_answer = """{"tool_call": {"name": "calculate", "arguments": {"expression": "(123 + 456) * 7"}}}"""

# Auto-check
assert your_answer != 'TODO: replace this placeholder with your code', 'Please replace the placeholder before running the assertion.'

try:
    parsed = json.loads(your_answer)
except json.JSONDecodeError as e:
    raise AssertionError(f"JSON format error: {e}")

assert "tool_call" in parsed, "The top level of the JSON should have a tool_call field"
assert parsed["tool_call"]["name"] == "calculate", "Should call the calculate tool"
assert "expression" in parsed["tool_call"]["arguments"], "arguments should contain an expression field"
# Check that the expression contains the required parts (operation order not strictly enforced)
expr = parsed["tool_call"]["arguments"]["expression"]
for token in ["123", "456", "7"]:
    assert token in expr, f"The expression should contain {token}"

print("Exercise 1 passed: you understand the JSON structure of a tool call")
print(f"Your answer: {parsed}")


### Exercise 2: Loss mask of a training sample

Below is the conversation flow of a training sample (simplified). For each message, decide whether the corresponding tokens should count toward the loss during training.

```text
[1] system:           "You can use the calculate(expression) tool"
[2] user:             "What is 12 x 8?"
[3] assistant(tool call): {"name": "calculate", "arguments": {"expression": "12*8"}}
[4] tool:             {"result": 96}
[5] assistant:        "12 x 8 equals 96."
```

Write the mask for each message as a list (True counts toward loss, False does not).

**Hint**: only positions the model needs to "generate" should count toward loss. Input positions (system/user/tool result) do not.


In [ ]:
# Exercise 2: loss mask of a training sample

# TODO: replace the placeholder list below with your answer
# Order corresponds to [1] system, [2] user, [3] assistant(tool call), [4] tool, [5] assistant
# Each element is True (counts toward loss) or False (does not)
your_mask = [False, False, True, False, True]

# Auto-check
assert your_mask != [None, None, None, None, None], 'Please replace the placeholder before running the assertion.'
assert len(your_mask) == 5, "There should be 5 elements"
assert all(isinstance(x, bool) for x in your_mask), "Each element should be True or False"

# Correct answer: system/user/tool are inputs (False); assistant is an output (True)
expected = [False, False, True, False, True]
assert your_mask == expected, "Incorrect mask: only assistant output positions contribute loss"

print("Exercise 2 passed: you understand the loss mask for Function Call training")
print(f"Your answer: {your_mask}")
print("Interpretation: [1]system [2]user are inputs; [3]assistant's tool call is content generated by the model;")
print("      [4] tool is an external result; [5] assistant final response is generated by the model.")


### Exercise 3: Parallel or sequential

For the three scenarios below, should each use parallel calls or sequential calls?

Scenario A: the user asks "What is the temperature in Beijing and in Shanghai today?"

Scenario B: the user asks "First check today's weather in Beijing, then recommend an outfit based on the temperature"

Scenario C: the user asks "What is 123 + 456? What is 789 - 123?"

Write the answer as a list where each element is the string `"parallel"` or `"sequential"`, in order A/B/C.

**Hint**: the criterion is whether later tool calls depend on earlier tool results. Dependency -> sequential; no dependency -> parallel.


In [ ]:
# Exercise 3: parallel or sequential

# TODO: replace the placeholder list below with your answer
# Order corresponds to scenarios A/B/C; each element is "parallel" or "sequential"
your_answer = ["parallel", "serial", "parallel"]

# Auto-check
assert your_answer != [None, None, None], 'Please replace the placeholder before running the assertion.'
assert len(your_answer) == 3, "There should be 3 elements"
assert all(x in ["parallel", "sequential"] for x in your_answer), "Each element should be 'parallel' or 'sequential'"

# Correct answers:
# A: Beijing and Shanghai weather are independent of each other -> parallel
# B: outfit depends on the weather result -> sequential
# C: two independent computations -> parallel
expected = ["parallel", "sequential", "parallel"]

# Compare one by one and give feedback
for i, (y, e, scene) in enumerate(zip(your_answer, expected, ["A", "B", "C"])):
    if y != e:
        hint = {
            "A": "Hint: Beijing and Shanghai weather are independent, no dependency",
            "B": "Hint: the clothing suggestion depends on the weather result",
            "C": "Hint: two independent arithmetic problems, no dependency"
        }[scene]
        raise AssertionError(f"Scenario {scene} is wrong. {hint}")

print("Exercise 3 passed: you can distinguish parallel from sequential calls")
print(f"Your answer: {your_answer}")
print("Interpretation:")
print("  A: two independent queries that can be called together -> parallel")
print("  B: the second call depends on the first result, so execute sequentially")
print("  C: two independent computations -> parallel")
